# Advisor reproducibility — one notebook, one configuration cell

Choose the basis source in **MODE**, then Run All. The same downstream cells
use the resulting basis labeled **Learned** for channels, complex effective
matrices and essential inline figures and the requested spectral/coupling views, alongside OTFS and AFDM.

- **load_existing**: implemented; load the verified basis for the chosen N (64,128,256).
- **train_new**: **UNAVAILABLE** in this branch. Selecting it raises a clear
  error before loading a substitute basis, provisioning data or starting work.
  Training belongs to the separately authorized follow-on branch; it is not
  implemented, simulated or started here.

**EVIDENCE_MODE is separate from basis selection:** saved_replay loads completed
BER counts (no campaign rerun); fresh generates new DD channels and computes a
bounded BER check using the already selected basis. It never means training.
Selection provenance/limits remain documented; internal studies stay separate.

After changing MODE, clear old outputs and restart the kernel before Run All.
An unavailable selection produces no new downstream results.


In [ ]:
# CONFIGURATION — edit this cell, then Run All from the top.
from pathlib import Path
import os, sys
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "advisor").is_dir() and (p / "pyproject.toml").is_file())
sys.path.insert(0, str(ROOT))
for key in ("OPENBLAS_NUM_THREADS", "OMP_NUM_THREADS", "VECLIB_MAXIMUM_THREADS"):
    os.environ[key] = "1"
os.environ["MPLCONFIGDIR"] = str(ROOT / "local/.matplotlib")

MODE = "load_existing"                # "load_existing" or "train_new" (UNAVAILABLE; follow-on only)
EVIDENCE_MODE = "saved_replay"         # "saved_replay" or "fresh"; independent of basis source
SCALING_ARTIFACT_DIR = ROOT / "local/scaling-artifacts"
SCALING_BUNDLE = ROOT / "local/advisor-scaling-artifacts.tar.gz"
SAVE_RUN = True                        # Optional local archive; figures are inline either way.

N = 64  # Available: 64/K24 historical; 128/K48 and 256/K96 one fresh nominal trial each.
MODEL = dict(n=N, paths=3, delay_range=(0.0, 4.0), doppler_range=(-0.75, 0.75))
BASIS_COLUMNS = {64:(0, 11, 15, 42, 44, 52), 128:(0, 8, 22, 46), 256:(0, 40, 87, 148)}
# Fixed columns: N64 previously frozen; higher N lowest native index per discovered group.
# First column also selects transmit j for the coupling-to-received-spectrum view.
# Example native IDs: edit the active N tuple; up to two per group, eight total.
# Display order follows the frozen coupling permutation, not this tuple order.
EXAMPLE_COLUMNS = {64:(0, 15, 44), 128:(0, 8, 22, 46), 256:(0, 40, 87, 148)}
DISPLAY = dict(basis_columns=BASIS_COLUMNS.get(N, (0,)),
               example_columns=EXAMPLE_COLUMNS.get(N, (0,)), snr_max_db=25.0)  # None = full SNR range.

# Readability alternatives. Each overlay includes all selected curves together.
# Defaults chosen by native index, independent of appearance/BER; edit native IDs.
OVERLAY8_COLUMNS = tuple(range(0, N, max(1, N//8)))  # Exactly eight unique IDs.
HALF_COLUMNS = tuple(range(0, N, 2))                 # Exactly N/2 unique IDs.
PLOT_VARIANTS = ("overlay8_complex", "overlay_half_complex",
                 "overlay8_energy_magnitude", "overlay_half_energy_magnitude",
                 "grouped_spectra")
REVIEW = dict(variants=PLOT_VARIANTS, overlay8_columns=OVERLAY8_COLUMNS, half_columns=HALF_COLUMNS)

FRESH = None                           # Must stay None in saved_replay.
if EVIDENCE_MODE == "fresh":
    FRESH = dict(seed=20261116, matrix_channels=2, matrix_index=0,
                 ber_channels=8, blocks_per_channel=2, snrs_db=(0.0, 10.0, 18.0, 25.0))


## 1. Select one basis, validate settings and prepare inputs

N selects its **own** Learned matrix and support: 64/K24 (historical), 128/K48 or
256/K96 (one completed nominal validation-selected trial each). No resizing,
N64 substitution or choosing a different winner per plot. K/N=3/8 throughout.
Higher-N training completed in the source study; this package does not train.

Replay uses the completed scaling release's nominal P3, delay[0,4] samples,
Doppler[-.75,.75] DFT bins and raw-sliced fixed/full curves. It is distinct from
the earlier focused N64 campaign, which remains internally available.
Fresh permits P1–6, delay within[0,8], Doppler within[-2,2],1–16 matrix channels,
2–64 BER channels,1–16 blocks,2–8 SNR points; channel×block×SNR product≤4096.
Changing evaluation controls does not train W or recalibrate its nominal support/groups.

The supplied local scaling bundle is automatically provisioned/verified. Missing size-specific
evidence fails; no source imports/downloads. CPU float64/complex128, one thread.
A future trainer must return W, its own fixed-K support and provenance into the
same downstream cells. **train_new is unavailable** and fails before any loading.


In [ ]:
from IPython.display import display, JSON
import pandas as pd
from advisor.notebook_workflow import start_workflow

workflow = None  # Clear prior run: selection failure must not reuse an old basis.
workflow = start_workflow(mode=MODE, evidence_mode=EVIDENCE_MODE, model=MODEL, display=DISPLAY, fresh=FRESH,
                          workspace=ROOT, save_run=SAVE_RUN,
                          scaling_artifacts=SCALING_ARTIFACT_DIR, scaling_bundle=SCALING_BUNDLE, review=REVIEW)
display(JSON(workflow.summary(), expanded=True))


## 2. Physical channel and complex effective matrices

DD convention: negative delay/Doppler signs, delay first;
Hraw=ΣhₚΔ(νₚ)Π(τₚ), H=Hraw/√(‖Hraw‖F²/N), A=WᴴHW.
Replay reconstructs structural channel index0 (fixed before BER, no outcome selection) from its paths; fresh mode
uses the configured seed/model and displayed channel index. This matrix bank and
the fresh BER bank have separate RNG namespaces. H, W and A remain available as
complex tensors in memory for inspection; the heatmap shows entry power only.


In [ ]:
channels, A = workflow.generate_channels()
H, Hraw, W = channels.h, channels.raw, workflow.ws
display(workflow.matrix_summary())
i = workflow.matrix_index
display(pd.DataFrame({"delay": channels.tau[i].numpy(), "Doppler": channels.nu[i].numpy(),
                      "complex gain": channels.gains[i].numpy()}))
print("Channel normalization:", float(channels.scale[i]))
workflow.show("channel_matrices_native")


## 3. Basis spectra and coupling-derived groups

Same native column IDs across time/frequency and waveforms; panel-title colors
identify **Learned's own frozen groups**, not baseline groups. Solid/dashed curves
show real/imaginary coefficients. Frequency curves sample the finite-block complex
DTFT S_j(f)=ΣₙW[n,j]exp(-j2πfn)/√N with eightfold zero padding, no window or peak
normalization. At DFT knots S_j(k/N)=C[k,j], C=FW. Magnitude is |C|; power is |C|²;
Σₖ|C[k,j]|²=1. Dense DTFT samples are correlated interpolation, not new evidence.

The following permutation table explicitly maps plotted ranks to native basis IDs.
Groups were inferred from **nominal coupling**, independently of this sample channel;
N64 has3 groups, N128/N256 each4. Freeze them across all views. They are approximate,
leaky and may occupy split/wrapped frequency regions, not a unique analytical construction.

A[i,j] couples transmitted basis j into received basis i: Hw_j=Σᵢw_i A[i,j].
The group view separates one |A|² sample, mean |A|² over512 nominal multipath channels,
and Q=mean|WᴴT(τ,ν)W|² over2048 nominal unitary atoms. Q is not the normalized
multipath second moment. Neither statistic is |mean(complex A)|².
Frozen nominal statistics remain labeled nominal even in fresh/model-changed mode.

The complex bridge Cᴴ(FHFᴴ)C=A is checked below; power spectra alone cannot reconstruct A.
The propagation view uses DISPLAY's first native column j and the same H as the channel
figure: received-index powers and coherently synthesized group spectra locate the leakage.
Group spectral **powers are not additive**: complex contributions have cross terms.


**Grouped energy/magnitude view (additional, not a replacement for complex spectra):**
one row per actual coupling group, matching the group colors/permutation/native-ID table.
Left: **every member**'s absolute N-point DFT coefficient energy |C[k,j]|², with
black arithmetic group mean. No individual peak normalization on this side; each
unit-norm column sums to1 over N DFT bins. Right: every member's finite-block
spectral **magnitude**, 20log₁₀(|S_j(f)|/max_f|S_j(f)|), individually peak-normalized
on the displayed32N-point grid and clipped at−60dB. Shapes here do not compare
absolute strength. Both axes use periodic f/Fs in[0,1); the endpoint1 labels the
wrap boundary, not an extra sample. Zero padding interpolates the finite-block
spectrum; it adds no physical frequency resolution. No window or retraining.
Rows/counts come from the frozen size-specific grouping, not the reference example.
The separate complex real/imaginary frequency figure above is retained.


**Individually identifiable examples:** EXAMPLE_COLUMNS in the single configuration
cell selects native Learned column IDs, independently of DISPLAY.basis_columns.
The selected examples are shown in frozen coupling order, at most two per group,
with an inline table linking **group, native j and permuted rank**. Defaults use
the lowest native ID of each frozen group; they are not selected for BER or appearance.
Black labeled lines in the grouped plot's right column identify those same examples
against all member curves. Each has its own row below: real/imaginary time basis,
readable spectral shape, and that exact transmitted column's |A[i,j]|² across the
received bases using the same displayed H. Titles repeat group/native j/rank;
received-basis bar colors match the frozen groups. Changing examples changes only
what is displayed; it does not change W, H, A, group inference, receiver or BER.
The separate actual complex-frequency comparison remains above.


**Readability variations for user review before final simplification:** PLOT_VARIANTS
chooses the enabled alternatives in this same configuration cell. OVERLAY8_COLUMNS
selects exactly8 native basis IDs; HALF_COLUMNS selects exactlyN/2 (32/64/128).
Every selected curve is overlaid together in each applicable panel, not split into
small subsets. Display follows the frozen coupling permutation; the full curve
key and inline index tables identify native j, group G and rank r. Color is a
stable native-ID color, not the group-color palette used in the group overview.
Line style is fixed by coupling rank. Default index choices are deterministic,
independent of appearance/BER. Dense overlays are retained for readability review;
use the labeled individual examples when curves overlap too heavily to trace.

The complex layouts separate real/imaginary time samples and actual complex DTFT
coefficients, without peak normalization. Energy/magnitude layouts instead show
absolute N-point |C|² and individually peak-normalized20log10 magnitude (−60dB
floor). Same32N-point interpolation grid; zero padding adds no physical resolution.
The coupling-group reference view remains an enabled alternative by default, and
all earlier coupling/individual/complex-basis figures remain available. These are
views of one Learned matrix per N, not comparisons among learned variants.


In [ ]:
workflow.show("basis_time")
workflow.show("basis_frequency")
groups = workflow.group_table()
with pd.option_context("display.max_rows", N):
    display(groups)
spectral = workflow.coupling_data()
print("max |WᴴW-I|:", spectral["unitarity_max"], "max |CᴴH_fC-A|:", spectral["bridge_max"])
workflow.show("coupling_groups")
display(JSON(workflow.review_summary(), expanded=True))
with pd.option_context("display.max_rows", N):
    display(workflow.overlay_table("overlay8"))
    display(workflow.overlay_table("half"))
workflow.show_variations()
display(workflow.example_table())
workflow.show("basis_examples")
workflow.show("coupling_propagation")


## 4. BER versus SNR — read the mode label

**Replay loads saved counts/intervals; it does not simulate BER.** Fresh mode
computes new paired H/bits/time-domain-noise observations for every method, with
the configured budgets. Fresh curves are smoke results with descriptive channel-SE
bands; they are not the saved campaign's bootstrap intervals or precise performance claims.

All saved nominal curves use4768channels/4,882,432bits per arm/point,
with2000 channel-bootstrap resamples. Budget-censored after complete minimum coverage;
some curves have fewer than300errors, flagged in table. Approximate pointwise intervals
do not account fully for adaptive stopping; one trial per higher N does not estimate
training-seed uncertainty. No universal/scaling-law claim.

Raw QPSK/Gray16QAM slicing, Es/N0. Fixed and full receivers stay separate;
actual observations always use full A. Fixed K=3N/8 positions come from the selected
basis provider (currently the verified nominal discovery masks), even when the fresh channel model changes: no hidden
support recalibration or training. Zero errors are unresolved; the table retains
counts, resolution and conservative independent-channel upper bounds.


In [ ]:
ber = workflow.evaluate_ber()
print(workflow.provenance["BER_status"])
display(ber[["modulation", "snr_db", "waveform", "receiver", "channels", "bits", "errors", "ber",
             "ci_low", "ci_high", "zero_cluster_upper95", "detection_limit"]])
display(ber.filter(items=["N", "K", "scenario", "snr_db", "waveform", "receiver", "modulation", "stop", "precision_target_met_for_curve"]))
workflow.show("ber_QPSK_raw")
workflow.show("ber_16QAM_raw")


## Complete

All enabled core and review figures are visible above (13 with default PLOT_VARIANTS). The optional archive preserves complex
matrices, complex spectral bridge, nominal Q/mean powers, groups/permutation, paths, masks, seeds, counts, settings and source hashes under local/;
you do not need to open it to review the results. Re-running creates a new archive
rather than overwriting evidence. After changing configuration, Run All again.
train_new remains unavailable until the separately authorized follow-on branch.
No training or new research workflow starts here.


In [ ]:
display(JSON(workflow.finish(), expanded=True))
